In [ ]:
import requests
import os
import re
import time
from dotenv import load_dotenv

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
load_dotenv()

#### Scopus API Search

- Scopus api has 2 views, complete (25 results) and standard (200 results)
- 5000 item total result limit without ‘cursor pagination’
- weekly quota 20000
- requests per sec 9 
- view results fields ![view results][https://dev.elsevier.com/sc_search_views.html]

we can query by publishers
```
'PUBLISHER(Wiley) AND TITLE-ABS(poly*)'
```
As of 15 Jun 26:
- For Wiley: 275551 results
- Springer Nature: 20022 results
- Elsevier: 766550 results

using polymer* OR copolymer*
As of 25 Jun 26
- For Wiley: 198739 results
- Springer Nature: 15268 results
- Elsevier: 510466 results

TODO: maybe use the COMPLETE view to retrieve as much abstracts as possible?

In [ ]:
publisher = "Elsevier"

url = "https://api.elsevier.com/content/search/scopus"
scopus_api_key = os.getenv("SCOPUS_API_KEY")
query = f'PUBLISHER({publisher}) AND TITLE-ABS-KEY(polymer* OR copolymer*)'
# query = f'PUBLISHER({publisher}) AND TITLE-ABS-KEY(polymer*)'

# need dc:description for abstract
FIELDS = "eid,prism:doi,dc:title,prism:publicationName,dc:publisher,openaccess,openaccessFlag"
# pagination does not need start param
# check cursor:@next for next page, and cursor:"*" for first page
params = {
    "query":   query,
    "field":   FIELDS,
    "start":   0,
    "count":   20,
    "view":    "STANDARD",   
    # "cursor": "*",
}
headers = {
    "X-ELS-APIKey": scopus_api_key,
    "Accept": "application/json"
}

response = requests.get(url, params=params, headers=headers)

assert response.status_code == 200, f"Request failed, ELS Status: {response.headers.get('X-ELS-Status')}"

data = response.json()

In [ ]:
data

In [ ]:
response.headers

In [ ]:
def fetch_scopus_year(
    publisher: str,
    pubyear: int,
    count: int = 200
) -> list[dict]:
    """Fetches papers from Scopus for a given publisher and publication year.

    Args:
        publisher (str): The name of the publisher to filter by.
        pubyear (int): The publication year to filter by.
        count (int, optional): The number of results to fetch per request. Defaults to 200.
        query_term (str, optional): The search term to use in the query. Defaults to "poly*".

    Returns:
        list[dict]: A list of dictionaries containing paper information.
    """
    url = "https://api.elsevier.com/content/search/scopus"
    scopus_api_key = os.getenv("SCOPUS_API_KEY")
    query = f'PUBLISHER({publisher}) AND TITLE-ABS(poly*) AND PUBYEAR < {pubyear}'

    # need dc:description for abstract
    FIELDS = "eid,prism:doi,dc:title,prism:publicationName,dc:publisher,openaccess,openaccessFlag"
    # pagination does not need start param
    # check cursor:@next for next page, and cursor:"*" for first page
    params = {
        "query":   query,
        "field":   FIELDS,
        "start":   0,
        "count":   200,
        "view":    "STANDARD",   
        # "cursor": "*",
    }
    headers = {
        "X-ELS-APIKey": scopus_api_key,
        "Accept": "application/json"
    }

    response = requests.get(url, params=params, headers=headers)

    assert response.status_code == 200, f"Request failed, ELS Status: {response.headers.get('X-ELS-Status')}"

    data = response.json()

    entries = data.get("search-results", {}).get("entry", [])

    records = []

    for e in entries:
        records.append({
            "eid": e.get("eid"),
            "doi": e.get("prism:doi"),
            "title": e.get("dc:title"),
            "journal": e.get("prism:publicationName"),
            "publisher": e.get("dc:publisher") or publisher,
            "open_access": e.get("openaccess"),
            "open_access_flag": e.get("openaccessFlag"),
        })

    if not records:
        print(f"No records found for publisher: {publisher}, year: {pubyear}")

    if len(records) < count:
        print(f"Only {len(records)} records found for publisher: {publisher}, year: {pubyear}")

    return records

In [ ]:
publisher = "Elsevier"
years = [2026, 2024, 2022, 2020, 2018]

all_records = []

for year in years:
    records = fetch_scopus_year(
        publisher=publisher,
        pubyear=year,
        count=200
    )

    all_records.extend(records)

    time.sleep(0.2)  # Sleep to respect API rate limits

df = pd.DataFrame(all_records)

# Optional deduplication, useful if Scopus returns overlaps
df = df.drop_duplicates(subset=["eid"], keep="first").reset_index(drop=True)

print(f"Total records fetched: {len(df)}")

In [ ]:
df

In [ ]:
output_path = os.path.join("..", "data", f"{publisher}_papers.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

At the moment we have extracted 1000 articles each from 3 publishers. For full extraction I need to implement cursor pagination.

In [ ]:
papers = ["elsevier_papers.csv", "springer_nature_papers.csv", "wiley_papers.csv"]
dfs = []
for paper in papers:
    path = os.path.join("..", "data", paper)
    df = pd.read_csv(path, sep="\t")
    dfs.append(df)
    print(f"{paper}: {len(df)} records")

df = pd.concat(dfs, ignore_index=True)

In [ ]:
df.head()

In [ ]:
print(f"OA papers: {len(df[df['open_access_flag'] == True])} of {len(df)}")

In [ ]:
unique_journals_per_publisher = (
    df.groupby("publisher")["journal"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="unique_journal_count")
)

print(unique_journals_per_publisher)

In [ ]:
journal_dist = (
    df.groupby(["publisher", "journal"])
    .size()
    .reset_index(name="paper_count")
    .sort_values(["publisher", "paper_count"], ascending=[True, False])
)

top_n = 20
top_journals = journal_dist.groupby("publisher", as_index=False).head(top_n)

publishers = top_journals["publisher"].unique()
fig, axes = plt.subplots(len(publishers), 1, figsize=(12, 4 * len(publishers)), constrained_layout=True)

if len(publishers) == 1:
    axes = [axes]

for ax, pub in zip(axes, publishers):
    subset = top_journals[top_journals["publisher"] == pub].sort_values("paper_count")
    ax.barh(subset["journal"], subset["paper_count"])
    ax.set_title(f"Top {top_n} journals for {pub}")
    ax.set_xlabel("Paper count")

plt.show()

Get abstracts

In [ ]:
df["abstract"] = "Not Found"
df.head()

In [ ]:
from difflib import SequenceMatcher

def normalize_title(title: str) -> str:
    title = title.lower()

    # remove punctuation
    title = re.sub(r"[^\w\s]", " ", title)

    # collapse whitespace
    title = re.sub(r"\s+", " ", title)

    return title.strip()

def title_match(
    title1: str,
    title2: str,
    threshold: float = 0.90,
) -> bool:
    t1 = normalize_title(title1)
    t2 = normalize_title(title2)

    score = SequenceMatcher(None, t1, t2).ratio()

    return score >= threshold

Need to add some keywords extraction here.

In [ ]:
def get_crossref_abstract(
    doi: str | None,
    title: str | None,
    mailto: str | None = None,
    timeout: int = 30,
) -> str | None:
    """
        get
    """
    doi = str(doi).strip()

    mailto = "kevinge@chalmers.se"
    headers = {
        "User-Agent": f"abstract-fetcher/0.1 (mailto:{mailto})",
    }

    url = f"https://api.crossref.org/works/{doi}"
    response = requests.get(
        url.format(doi=doi),
        headers=headers,
        timeout=timeout,
    )

    response.raise_for_status()

    data = response.json()

    title_list = data.get("message", {}).get("title", [])
    crossref_title = title_list[0] if title_list else None
    safe_flag = False
    if crossref_title:
        if title_match(title, crossref_title):
            safe_flag = True
        else:
            print(f"[Warning]: Title Mismatch for doi: {doi}")
            print(f"Title: {title}")
            print(f"crossref: {crossref_title}")

    raw_abstract = data.get("message", {}).get("abstract")

    if not raw_abstract:
        return None, None

    # Strip JATS XML tags (e.g. <jats:p>, <jats:title>) and collapse whitespace
    # Remove <jats:title>Abstract</jats:title>
    clean_abstract = re.sub(
        r"<jats:title[^>]*>.*?</jats:title>",
        " ",
        raw_abstract,
        flags=re.I | re.S,
    )
    clean_abstract = re.sub(r"<[^>]+>", " ", clean_abstract)
    clean_abstract = re.sub(r"\s+", " ", clean_abstract).strip()

    return clean_abstract, safe_flag

In [ ]:
for idx, row in df.iterrows():

    current_abstract = row["abstract"]

    # Skip rows that already have an abstract
    if (
        pd.notna(current_abstract)
        and str(current_abstract).strip()
        and str(current_abstract).strip().lower() != "not found"
    ):
        continue

    doi = row["doi"]
    title = row["title"]

    if pd.isna(doi):
        print(f"[CHECK]: doi not in dataframe at index {idx}")
        continue

    time.sleep(0.1)  # be nice to Crossref

    try:
        abstract, safe_flag = get_crossref_abstract(
            doi=doi,
            title=title,
            mailto = "kevinge@chalmers.se"
        )

    except Exception as e:
        print(
            f"Crossref failed | DOI={doi} | "
            f"TITLE={title[:80]} | ERROR={e}"
        )
        continue

    if not abstract:
        continue

    if not safe_flag:
        continue

    df.at[idx, "abstract"] = abstract

    print("[INFO]: Abstract found")
    if idx % 100 == 0 and idx != 0:
        print(f"[INFO]: currently at idx {idx}")

In [ ]:
summary = (
    df.assign(
        abstract_found=lambda x: (
            x["abstract"].notna()
            & (x["abstract"] != "Not Found")
            & (x["abstract"].str.strip() != "")
        )
    )
    .groupby("publisher")
    .agg(
        total_papers=("publisher", "size"),
        abstracts_found=("abstract_found", "sum"),
    )
)

summary["retrieval_rate"] = (
    100 * summary["abstracts_found"] / summary["total_papers"]
).round(1)

print(summary)

In [ ]:
output_path = os.path.join("..", "data", f"papers_post_crossref.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

Use Scopus

In [ ]:
output_path = os.path.join("..", "data", f"papers_post_crossref.csv")
df = pd.read_csv(
    output_path, sep="\t"
)

In [ ]:
df.head()

In [ ]:
from pybliometrics.scopus import AbstractRetrieval

# TODOs: we need some sort of keyword extraction here
def get_scopus_abstract(
    doi: str | None,
) -> str | None:

    if not doi:
        return None

    doi = str(doi).strip()

    try:
        record = AbstractRetrieval(
            doi,
            id_type="doi",
            view="META_ABS",
        )

        # Prefer abstract
        abstract = record.abstract

        # Fallback to description if abstract missing
        if not abstract:
            abstract = getattr(record, "description", None)

        if not abstract:
            print(
                f"No abstract/description found for DOI: {doi}"
            )
            return None

        abstract = " ".join(str(abstract).split())

        return abstract

    except Exception as e:
        print(
            f"Scopus AbstractRetrieval failed | DOI={doi} | "
            f"ERROR={type(e).__name__}: {e}"
        )
        return None

In [ ]:
for idx, row in df.iterrows():
    if idx % 100 == 0 and idx != 0:
        print(f"[INFO]: currently at idx {idx}")

    current_abstract = row["abstract"]

    # Skip rows that already have an abstract
    if (
        pd.notna(current_abstract)
        and str(current_abstract).strip()
        and str(current_abstract).strip().lower() != "not found"
    ):
        continue

    doi = row["doi"]

    if pd.isna(doi):
        print(f"[CHECK]: doi not in dataframe at index {idx}")
        continue

    abstract = get_scopus_abstract(
        doi=doi
    )
    time.sleep(0.2)  # respect api rate limits

    if not abstract:
        continue

    df.at[idx, "abstract"] = abstract

    print("[INFO]: Abstract found")

In [ ]:
summary = (
    df.assign(
        abstract_found=lambda x: (
            x["abstract"].notna()
            & (x["abstract"] != "Not Found")
            & (x["abstract"].str.strip() != "")
        )
    )
    .groupby("publisher")
    .agg(
        total_papers=("publisher", "size"),
        abstracts_found=("abstract_found", "sum"),
    )
)

summary["retrieval_rate"] = (
    100 * summary["abstracts_found"] / summary["total_papers"]
).round(1)

print(summary)

In [ ]:
# output_path = os.path.join("..", "data", f"papers_post_scopus.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

In [ ]:
# Publisher TDMs

Building a classifier

In [ ]:
import os
from typing import Optional

from pydantic import BaseModel, Field, ConfigDict
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

from langchain_core.output_parsers import PydanticOutputParser


class PolymerSynthesisClassification(BaseModel):
    model_config = ConfigDict(extra="ignore")

    is_experimental_polymer_synthesis: bool = Field(
        description=(
            "True if the paper reports experimental synthesis, preparation, "
            "fabrication, modification, or degradation of a polymer/material by "
            "the authors in this study. False for reviews, purely computational "
            "papers, characterization-only papers, application-only papers, "
            "biological studies without polymer synthesis, or papers where the "
            "polymer is only purchased/used."
        )
    )


SYSTEM_PROMPT = """
You are a scientific paper classifier for polymer synthesis datasets.

Classify whether the given paper is relevant for an experimental polymer synthesis extraction pipeline.

Return true ONLY if the title and abstract indicate that the authors experimentally synthesized, prepared, fabricated, modified, functionalized, degraded, crosslinked, polymerized, copolymerized, grafted, or otherwise made a polymer/material in this study.

Return false if the paper is:
- a review, perspective, survey, book chapter, editorial, or commentary
- purely computational, theoretical, modeling, simulation, or data-mining
- only about characterization/testing of an existing polymer
- only about applications of a purchased/commercial polymer
- only about biological/medical/environmental testing with no polymer synthesis
- only about monomer synthesis without polymer synthesis
- only about small molecules, catalysts, membranes, adsorbents, or composites where no polymer is synthesized/prepared by the authors
- unclear from the title and abstract

Important:
- Polymer synthesis papers often mention polymerization, copolymerization, grafting, crosslinking, curing, RAFT, ATRP, ROMP, ring-opening polymerization, anionic polymerization, condensation polymerization, or preparation of polymer networks.
- Characterization such as NMR, GPC/SEC, DSC, DMA, TGA, FTIR, mechanical testing, or morphology may support relevance, but characterization alone is not enough.
- Be conservative. If experimental polymer synthesis by the authors is not clear, return false.
"""


USER_PROMPT = """
Title:
{title}

Abstract:
{abstract}

Format instructions:
{format_instructions}
"""

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1")


llm = ChatOpenAI(
    # model="openai/gpt-4o-mini",  # change to any OpenRouter model you prefer
    # model="openai/gpt-oss-120b:free",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
    temperature=0,
)

# classifier = llm.with_structured_output(PolymerSynthesisClassification)

parser = PydanticOutputParser(
    pydantic_object=PolymerSynthesisClassification
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("user", USER_PROMPT),
    ]
)
classification_chain = prompt | llm | parser

In [ ]:
def classify_paper(
    title: str,
    abstract: str | None = None,
) -> bool:
    """
    Return True if the paper appears to contain experimental polymer synthesis
    performed by the authors, otherwise False.
    """

    try:
        result = classification_chain.invoke(
            {
                "title": title or "",
                "abstract": abstract or "",
                "format_instructions": parser.get_format_instructions(),
            }
        )

        return bool(
            result.is_experimental_polymer_synthesis
        )

    except Exception as e:
        print(
            f"Classification failed | "
            f"TITLE={title[:100] if title else 'None'} | "
            f"ERROR={type(e).__name__}: {e}"
        )

        return False

In [ ]:
labels = []

for idx, row in df.iterrows():
    if idx % 100 == 0 and idx != 0:
        print(f"[INFO]: currently at idx {idx}")

    title = row.get("title")
    abstract = row.get("abstract")

    # Missing title -> automatically False
    if not title or not str(title).strip():
        labels.append(False)
        continue

    # Missing abstract -> automatically False
    if (
        abstract is None
        or pd.isna(abstract)
        or str(abstract).strip() == ""
        or str(abstract).strip().lower() == "not found"
    ):
        labels.append(False)
        continue

    result = classify_paper(title, abstract)

    labels.append(result)

    time.sleep(3.8)

There are some structured output parsing errors. Reduced a lot when adding format instructions into the user prompt. But still very few errors. But that can be accomodated by instructor I think. But with instructor we can't do think steps

In [ ]:
assert len(df) == len(labels)
# df["is_experimental_polymer_synthesis"] = labels
df["polymer_synthesis_oss_120"] = labels

In [ ]:
# output_path = os.path.join("..", "data", f"papers_classified_w_oss.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

In [ ]:
from sklearn.metrics import cohen_kappa_score

agreement = (
    df["is_experimental_polymer_synthesis"]
    == df["polymer_synthesis_oss_120"]
).mean()

print(f"Agreement: {agreement:.2%}")

kappa = cohen_kappa_score(
    df["is_experimental_polymer_synthesis"],
    df["polymer_synthesis_oss_120"]
)

print(f"Cohen's κ = {kappa:.3f}")

In [ ]:
pd.crosstab(
    df["is_experimental_polymer_synthesis"],
    df["polymer_synthesis_oss_120"],
    rownames=["GPT4"],
    colnames=["OSS"],
)

In [ ]:
disagreement = df[
    df["is_experimental_polymer_synthesis"] != df["polymer_synthesis_oss_120"]
]

print(len(disagreement))

display_df = disagreement[
    [
        "title",
        "abstract",
        "is_experimental_polymer_synthesis",
        "polymer_synthesis_oss_120",
    ]
].copy()

display_df.columns = ["title", "abstract", "gpt", "oss"]

display(
    display_df.head().style.set_properties(
        subset=["title"],
        **{
            "white-space": "pre-wrap",
            "max-width": "300px",
        }
    ).set_properties(
        subset=["abstract"],
        **{
            "white-space": "pre-wrap",
            "max-width": "800px",
        }
    )
)

In [ ]:
disagreement["abstract"].iloc[0]

In [ ]:
# disagreement.to_csv(
#     os.path.join("..", "data", f"classifier_disagreements.csv"),
#     sep="\t",
#     index=False,
# )

In [ ]:
summary = (
    df.groupby("publisher")["is_experimental_polymer_synthesis"]
      .agg(
          total_papers="count",
          synthesis_papers="sum",
      )
)

summary["fraction"] = (
    summary["synthesis_papers"]
    / summary["total_papers"]
).round(3)

summary["percent"] = (
    100 * summary["fraction"]
).round(1)

print(summary)

In [ ]:
synthesis_df = df[
    df["is_experimental_polymer_synthesis"]
]
journal_dist = (
    synthesis_df.groupby(["publisher", "journal"])
    .size()
    .reset_index(name="paper_count")
    .sort_values(
        ["publisher", "paper_count"],
        ascending=[True, False],
    )
)

top_n = 20

top_journals = (
    journal_dist.groupby(
        "publisher",
        as_index=False,
    )
    .head(top_n)
)

publishers = top_journals["publisher"].unique()

fig, axes = plt.subplots(
    len(publishers),
    1,
    figsize=(12, 4 * len(publishers)),
    constrained_layout=True,
)

if len(publishers) == 1:
    axes = [axes]

for ax, pub in zip(axes, publishers):

    subset = (
        top_journals[
            top_journals["publisher"] == pub
        ]
        .sort_values("paper_count")
    )

    ax.barh(
        subset["journal"],
        subset["paper_count"],
    )

    ax.set_title(
        f"Top {top_n} synthesis journals for {pub}"
    )

    ax.set_xlabel(
        "Experimental polymer synthesis papers"
    )

plt.show()

In [ ]:
synthesis_df["journal"].value_counts().head(10)

In [ ]:
top_10_journals = (
    synthesis_df["journal"]
    .value_counts()
    .head(5)
    .index
)

random_papers = (
    synthesis_df[
        synthesis_df["journal"].isin(top_10_journals)
    ]
    .groupby("journal", group_keys=False)
    .sample(2, random_state=12)
)

random_papers[
    [
        "journal",
        "title",
        "doi",
        "is_experimental_polymer_synthesis",
        "polymer_synthesis_oss_120"
    ]
]

In [ ]:
journal_name = "Polymer"

journal_papers = synthesis_df[
    synthesis_df["journal"].str.lower() == journal_name.lower()
][
    [
        "title",
        "doi",
    ]
].copy()

journal_papers
display(
    journal_papers.style.set_properties(
        subset=["title"],
        **{
            "white-space": "pre-wrap",
            "max-width": "1000px",
        }
    )
)

In [ ]:
# randomly select a paper from the top 10 for screenshots
# This costed us 0.35 dollars
# We now also have prediction gpt oss 120b, the inference time of this model is very slow. Somewhere around 8 hours for 3000 datapoints

TDM full text retrival

In [ ]:
# Something to think about is word cloud on keywords. check if we can get keywords for classified and non classified models.

# Tried for Springer Nature papers only. total faliure. No keywords present

In [ ]:
springer_df = df[
    (df["publisher"] == "Springer Nature")
    & (
        (df["open_access_flag"] == True)
        | (df["open_access_flag"].astype(str).str.lower() == "true")
        | (df["open_access"].astype(str) == "1")
    )
].copy()

springer_df["springer_subjects"] = None
springer_df["springer_keywords"] = None

print(
    f"Found {len(springer_df)} Springer Nature OA papers"
)

In [ ]:
springer_df["springer_subjects"] = None
springer_df["springer_keywords"] = None

SPRINGER_NATURE_API_KEY = os.getenv("SPRINGER_NATURE_API_KEY")

for idx, row in springer_df.iterrows():

    if idx % 100 == 0 and idx != 0:
        print(f"[INFO] Currently at idx {idx}")

    doi = row.get("doi")

    if pd.isna(doi) or not str(doi).strip():
        print(f"[CHECK] Missing DOI at idx={idx}")
        continue

    doi = str(doi).strip()

    url = (
        "https://api.springernature.com/openaccess/json"
        f"?api_key={SPRINGER_NATURE_API_KEY}"
        f"&s=1&p=10&q=(doi:\"{doi}\")"
    )

    try:
        response = requests.get(
            url,
            timeout=30,
        )

        response.raise_for_status()

        data = response.json()

    except Exception as e:
        print(
            f"[ERROR] Springer API failed | "
            f"idx={idx} | DOI={doi} | {e}"
        )
        continue

    records = data.get("records", [])

    if not records:
        continue

    record = records[0]

    subjects = record.get("subjects", []) or []

    keywords = (
        record.get("keywords")
        or record.get("keyword")
        or []
    )

    if isinstance(keywords, str):
        keywords = [keywords]

    if isinstance(keywords, dict):
        keywords = list(keywords.values())

    print(keywords)

    springer_df.at[idx, "springer_subjects"] = "; ".join(subjects)
    springer_df.at[idx, "springer_keywords"] = "; ".join(keywords)

    time.sleep(1)

In [ ]:
subject_df = springer_df[
    springer_df["springer_subjects"].notna()
    & (springer_df["springer_subjects"].str.strip() != "")
].copy()

subject_df["subject"] = (
    subject_df["springer_subjects"]
    .str.split(";")
)

subject_df = subject_df.explode("subject")

subject_df["subject"] = subject_df["subject"].str.strip()

subject_df = subject_df[
    subject_df["subject"].notna()
    & (subject_df["subject"] != "")
]
subject_counts = (
    subject_df
    .groupby(["is_experimental_polymer_synthesis", "subject"])
    .size()
    .reset_index(name="count")
)
top_false = (
    subject_counts[
        subject_counts["is_experimental_polymer_synthesis"] == False
    ]
    .sort_values("count", ascending=False)
    .head(10)
)
top_true = (
    subject_counts[
        subject_counts["is_experimental_polymer_synthesis"] == True
    ]
    .sort_values("count", ascending=False)
    .head(10)
)

In [ ]:
top_true

In [ ]:
# after this we need to do other sources. PMC and so on (if time is not available we skip this and move forward)

In [ ]:
import requests
import os
import re
import time
from dotenv import load_dotenv

import pandas as pd
import matplotlib.pyplot as plt

EuroPMC which containes PMC, PubMed and others.

hits - 287986

In [ ]:
params = {
    "query": "(TITLE_ABS:polymer* OR TITLE_ABS:copolymer*) AND HAS_ABSTRACT:y AND (IN_PMC:y OR IN_EPMC:y OR HAS_PDF:y)",
    "format": "json",
    "resultType": "core",
    "pageSize": 1000,
}

r = requests.get(
    "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
    params=params,
)

results = r.json()["resultList"]["result"]

In [ ]:
results[0]

In [ ]:
r.json()["hitCount"]

In [ ]:
records = []

for item in results:

    journal_info = item.get("journalInfo") or {}
    journal = journal_info.get("journal") or {}

    records.append(
        {
            "doi": item.get("doi", None),
            "title": item.get("title", None),
            "pmcid": item.get("pmcid", None),
            "pmid": item.get("pmid", None),
            "abstract": item.get("abstractText", None),
            "authors": item.get("authorString", None),
            "authorList": item.get("authorList", None),
            "affiliation": item.get("affiliation", None),
            "journal": journal.get("title", None),
            "publication_year": item.get("pubYear", None),
            "is_open_access": (
                item.get("isOpenAccess") == "Y"
                if "isOpenAccess" in item
                else None
            ),
            "in_epmc": (
                item.get("inEPMC") == "Y"
                if "inEPMC" in item
                else None
            ),
            "in_pmc": (
                item.get("inPMC") == "Y"
                if "inPMC" in item
                else None
            ),
            "has_pdf": (
                item.get("hasPDF") == "Y"
                if "hasPDF" in item
                else None
            ),
        }
    )

df = pd.DataFrame(records)

In [ ]:
output_path = os.path.join("..", "data", f"EuroPMC_papers.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

arXiv hits - 15225

In [ ]:
import xml.etree.ElementTree as ET

search_query = (
    "(ti:polymer OR abs:polymer OR "
    "ti:polymers OR abs:polymers OR "
    "ti:copolymer OR abs:copolymer OR "
    "ti:copolymers OR abs:copolymers OR "
    "ti:polymerization OR abs:polymerization OR "
    "ti:copolymerization OR abs:copolymerization)"
)

url = "https://export.arxiv.org/api/query"

params = {
    "search_query": search_query,
    "start": 0,
    "max_results": 10,
    "sortBy": "submittedDate",
    "sortOrder": "descending",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

root = ET.fromstring(response.text)

In [ ]:
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

In [ ]:
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "opensearch": "http://a9.com/-/spec/opensearch/1.1/"
}

In [ ]:
root.find("opensearch:totalResults", ns).text

In [ ]:
def print_tree(elem, level=0):
    print("  " * level + elem.tag)
    for child in elem:
        print_tree(child, level + 1)

In [ ]:
print_tree(root)

In [ ]:
ns = {
    "atom": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

records = []

for entry in root.findall("atom:entry", ns):
    title = entry.findtext("atom:title", default=None, namespaces=ns)
    abstract = entry.findtext("atom:summary", default=None, namespaces=ns)

    # arXiv identifier
    arxiv_url = entry.findtext("atom:id", default=None, namespaces=ns)
    arxiv_id = arxiv_url.split("/")[-1] if arxiv_url else None

    authors = []
    affiliations = []

    for author in entry.findall("atom:author", ns):
        name = author.findtext("atom:name", default=None, namespaces=ns)
        affiliation = author.findtext("arxiv:affiliation", default=None, namespaces=ns)

        if name:
            authors.append(name)

        if affiliation:
            affiliations.append(affiliation)

    doi = entry.findtext("arxiv:doi", default=None, namespaces=ns)

    primary_category = None
    primary = entry.find("arxiv:primary_category", ns)
    if primary is not None:
        primary_category = primary.attrib.get("term")

    categories = [
        c.attrib.get("term")
        for c in entry.findall("atom:category", ns)
    ]

    for link in entry.findall("atom:link", ns):
        if link.attrib.get("title") == "pdf":
            pdf_url = link.attrib.get("href")
            break

    records.append({
        "arxiv_id": arxiv_id,
        "doi": doi,
        "title": " ".join(title.split()) if title else None,
        "abstract": " ".join(abstract.split()) if abstract else None,
        "authors": "; ".join(authors) if authors else None,
        "affiliations": "; ".join(dict.fromkeys(affiliations)) if affiliations else None,
        "published": entry.findtext(
                "atom:published",
                default=None,
                namespaces=ns,
            ),
        "primary_category": primary_category,
        "categories": "; ".join(categories),
        "abstract_url": arxiv_url,
        "pdf_url": pdf_url,
    })

In [ ]:
records[0]

In [ ]:
output_path = os.path.join("..", "data", f"arxiv_papers.csv")

# df.to_csv(
#     output_path,
#     sep="\t",
#     index=False,
#     encoding="utf-8",
# )

In [ ]:
ChemRxiv's primary OpenEngage API can be blocked by Cloudflare (HTTP 403) in some
environments. This module provides a fallback based on Crossref's public API
using the ChemRxiv DOI prefix (``10.26434``).

https://github.com/jannisborn/paperscraper/blob/main/paperscraper/get_dumps/utils/chemrxiv/crossref_api.py

https://connect.acspubs.org/chemrxiv-migration-FAQ

The pdf endpoint is willl not work because of Cloudflare JavaScript challenge

The work around is to start a browser and work it out: too much to do
from pathlib import Path
from playwright.sync_api import sync_playwright


In [ ]:
import requests

query = "polymer"
url = "https://api.crossref.org/works"
params = {
    # "query": query,
    "filter": "prefix:10.26434,type:posted-content",  # 10.26434 = ChemRxiv DOI prefix
    "select": "DOI,title,author,posted,abstract",
}
headers = {"User-Agent": "my-research-script/1.0 (mailto:you@example.com)"}

r = requests.get(url, params=params, headers=headers, timeout=30)
r.raise_for_status()


In [ ]:
r.json()

In [ ]:
papers = []
for item in r.json()["message"]["items"]:
    date_parts = (item.get("posted") or item.get("issued") or {}).get("date-parts", [[]])[0]
    pub_year = str(date_parts[0]) if date_parts else None

    authors = []
    for a in item.get("author", []):
        name = f"{a.get('given', '')} {a.get('family', '')}".strip()
        affs = [af["name"] for af in a.get("affiliation", []) if "name" in af]
        authors.append({"name": name, "affiliations": affs})

    papers.append({
        "doi":      item.get("DOI"),
        "title":    (item.get("title") or [""])[0],
        "abstract": item.get("abstract", ""),   # often empty — see note
        "pub_year": pub_year,
        "authors":  authors,
    })

In [ ]:
import requests

url = "https://chemrxiv.org/doi/pdf/10.26434/chemrxiv.12536153.v2"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0.0.0 Safari/537.36"
    )
}

r = requests.get(url, headers=headers, timeout=60)
print(r.status_code)
print(r.headers.get("content-type"))

In [ ]:
print(r.text[:500])

In [ ]:
import requests

doi = "10.26434/chemrxiv-2023-lw1ns"

r = requests.get(
    f"https://doi.org/{doi}",
    headers={"Accept": "text/html"},
    allow_redirects=True,
)

print(r.url)

In [ ]:
r = requests.get(r.url, timeout=60)
r.raise_for_status()

In [ ]:
import chemrxiv

In [ ]:
client = chemrxiv.Client()

In [ ]:
paper = client.item_by_doi(doi)
paper.download_pdf(dirpath='.', filename='test.pdf')

- 2851294 papers with "query": "polymer* | copolymer*"

In [ ]:
import os
import requests

API_KEY = os.getenv("S2_API_KEY")

headers = {
    "x-api-key": API_KEY
}

FIELDS = ",".join([
    "paperId",
    "title",
    "abstract",
    "year",
    "venue",
    "externalIds",
    "isOpenAccess",
    "openAccessPdf",
])

query = {
    "query": "polymer* | copolymer*",
    "fields": FIELDS,
    "openAccessPdf": "",
    'pdf': True,
}

response = requests.get(
    "http://api.semanticscholar.org/graph/v1/paper/search/bulk",
    headers=headers,
    params=query,
    timeout=60,
).json()

In [ ]:
response

In [ ]:
len(response["data"])

In [ ]:
response["data"][0]

In [ ]:
import os
from pathlib import Path
import requests

EMAIL = "kevin.george@gu.se"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/137.0 Safari/537.36"
    )
}

pdf_dir = Path("pdfs")
pdf_dir.mkdir(exist_ok=True)


def sanitize_filename(name):
    return (
        name.replace("/", "_")
            .replace("\\", "_")
            .replace(":", "_")
            .replace("?", "_")
    )


def try_download(url, outfile):
    try:
        r = requests.get(
            url,
            headers=headers,
            timeout=120,
            allow_redirects=True,
        )

        if (
            r.status_code == 200
            and "pdf" in r.headers.get("content-type", "").lower()
        ):
            outfile.write_bytes(r.content)
            print("Downloaded:", outfile.name)
            return True

        print(
            "failed",
            r.status_code,
            r.headers.get("content-type"),
            url,
        )
        return False

    except Exception as e:
        print("ERROR:", url, e)
        return False


def unpaywall_candidates(doi):
    try:
        r = requests.get(
            f"https://api.unpaywall.org/v2/{doi}",
            params={"email": EMAIL},
            timeout=60,
        )
        r.raise_for_status()

        data = r.json()

        candidates = []

        best = data.get("best_oa_location")
        if best:
            if best.get("url_for_pdf"):
                candidates.append(best["url_for_pdf"])
            if best.get("url"):
                candidates.append(best["url"])

        for loc in data.get("oa_locations", []):
            if loc.get("url_for_pdf"):
                candidates.append(loc["url_for_pdf"])
            if loc.get("url"):
                candidates.append(loc["url"])

        # remove duplicates while preserving order
        seen = set()
        unique = []
        for u in candidates:
            if u not in seen:
                unique.append(u)
                seen.add(u)

        return unique

    except Exception as e:
        print("Unpaywall failed:", doi, e)
        return []


for paper in response["data"]:

    doi = paper.get("externalIds", {}).get("DOI")
    paper_id = paper["paperId"]

    filename = sanitize_filename(doi or paper_id) + ".pdf"
    outfile = pdf_dir / filename

    if outfile.exists():
        continue

    candidates = []

    oa = paper.get("openAccessPdf")
    if oa and oa.get("url"):
        candidates.append(oa["url"])

    if doi:
        candidates.extend(unpaywall_candidates(doi))

    downloaded = False

    for url in candidates:
        if try_download(url, outfile):
            downloaded = True
            break

    if not downloaded:
        print(f"Could not download {doi}")

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}
u = requests.get(
    f"https://onlinelibrary.wiley.com/doi/pdfdirect/10.1002/sstr.202000144",
    headers=headers,
    timeout=60,
)
u.raise_for_status()
data = u.json()

In [ ]:
import requests

doi = "10.1002/sstr.202000144"
email = "kevin.george@gu.se"

u = requests.get(
    f"https://api.unpaywall.org/v2/{doi}",
    params={"email": email},
    timeout=60,
)
u.raise_for_status()
data = u.json()

best = data.get("best_oa_location") or {}
print(best.get("url_for_pdf"))
print(best.get("url"))
print(best.get("host_type"))
print(best.get("license"))

In [ ]:
candidates = []

for loc in [data.get("best_oa_location")] + data.get("oa_locations", []):
    if not loc:
        continue
    if loc.get("url_for_pdf"):
        candidates.append(loc["url_for_pdf"])
    elif loc.get("url"):
        candidates.append(loc["url"])

print(candidates)

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3',
}
def try_download(url, out):
    r = requests.get(
        url,
        headers=headers,
        timeout=120,
        allow_redirects=True,
    )
    if r.status_code == 200 and "pdf" in r.headers.get("content-type", "").lower():
        with open(out, "wb") as f:
            f.write(r.content)
        return True
    print("failed", r.status_code, r.headers.get("content-type"), url)
    return False

for i, url in enumerate(candidates):
    out = f"paper_{i}.pdf"
    if try_download(url, out):
        print("Downloaded:", out)
        break

In [ ]:
import os
import requests

API_KEY = os.getenv("S2_API_KEY")



headers = {
    "x-api-key": API_KEY
}

# 1. Get latest release
releases = requests.get(
    "https://api.semanticscholar.org/datasets/v1/release",
    headers=headers,
    timeout=60,
).json()

latest = releases[-1]
print("Latest release:", latest)

# # 2. Get S2ORC download links
# datasets = requests.get(
#     # f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/s2orc",
#     f"https://api.semanticscholar.org/datasets/v1/release/{latest}",
#     headers=headers,
#     timeout=60,
# ).json()

# files = dataset["files"]
# print("Number of shards:", len(files))

In [ ]:
response = requests.get(
    "https://api.semanticscholar.org/datasets/v1/release",
    headers=headers,
    timeout=60,
)

In [ ]:
response.headers

In [ ]:
response.json()

In [ ]:
datasets = requests.get(
    # f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/s2orc",
    f"https://api.semanticscholar.org/datasets/v1/release/{latest}",
    headers=headers,
    timeout=60,
).json()

In [ ]:
print(datasets["README"])

In [ ]:
for dataset in datasets["datasets"]:
    print(f"Dataset: {dataset['name']}")
    print(f"Description: {dataset['description']}")
    print(dataset["README"])

    print(" ")
    print("--"*100)
    print(" ")

In [ ]:
# 2. Get S2ORC download links
dataset = requests.get(
    f"https://api.semanticscholar.org/datasets/v1/release/{latest}/dataset/papers",
    headers=headers,
    timeout=60,
).json()

files = dataset["files"]
print("Number of shards:", len(files))

In [ ]:
dataset

In [ ]:
files

In [ ]:
from tqdm import tqdm
from urllib.parse import urlparse
from pathlib import Path

In [ ]:
LOCAL_PATH = os.path.join("..", "data", "s2orc_v2")
os.makedirs(LOCAL_PATH, exist_ok=True)

In [ ]:
for url in tqdm(files):
    shard_id = Path(urlparse(url).path).stem
    with requests.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(os.path.join(LOCAL_PATH, f"{shard_id}.gz"), "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    
print("Downloaded all shards.")